# GenAI ROI Analysis - Part 3: Analysis & Visualization

**Author:** [Your Name]  
**Date:** March 2025

---

## Overview

This notebook contains the final analysis and visualizations:
1. Adoption analysis by segment
2. Sentiment & trust analysis
3. Benefits & challenges analysis
4. ROI analysis
5. Dashboard creation

---

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

# Color palette
COLORS = {
    'primary': '#2E86AB',
    'secondary': '#A23B72',
    'success': '#28A745',
    'warning': '#FFC107',
    'danger': '#DC3545',
    'info': '#17A2B8'
}

# Load cleaned data
df = pd.read_csv('../data/processed/genai_roi_clean.csv')
print(f"Loaded {len(df):,} records")

## 2. Adoption Overview

In [ ]:
# Summary statistics
print("="*50)
print("AI ADOPTION OVERVIEW")
print("="*50)

total = len(df)
ai_users = (df['Is_AI_User'] == 1).sum()
adoption_rate = ai_users / total * 100

print(f"Total Respondents: {total:,}")
print(f"AI Users: {ai_users:,}")
print(f"Adoption Rate: {adoption_rate:.1f}%")

In [ ]:
# Adoption pie chart
fig, ax = plt.subplots(figsize=(8, 6))

adoption = df['AI_Status'].value_counts()
colors = [COLORS['success'], COLORS['danger'], COLORS['warning']]

wedges, texts, autotexts = ax.pie(
    adoption.values, 
    labels=adoption.index,
    autopct='%1.1f%%',
    colors=colors,
    explode=(0.02, 0.02, 0.02),
    startangle=90
)

plt.setp(autotexts, size=11, weight='bold', color='white')
ax.set_title(f'AI Tool Adoption Status\n(n={len(df):,})', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../visualizations/01_adoption_overview.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Adoption by Company Size

In [ ]:
# Calculate adoption by company size
adoption_size = df.groupby('Company_Size_Category')['Is_AI_User'].agg(['sum', 'count'])
adoption_size['rate'] = (adoption_size['sum'] / adoption_size['count'] * 100).round(1)

# Sort by size order
size_order = ['Solo/Freelancer', 'Micro (2-9)', 'Small (10-19)', 'Small-Mid (20-99)',
              'Mid (100-499)', 'Mid-Large (500-999)', 'Large (1K-5K)', 
              'Large (5K-10K)', 'Enterprise (10K+)']
adoption_size = adoption_size.reindex([s for s in size_order if s in adoption_size.index])

print("Adoption by Company Size:")
print(adoption_size)

In [ ]:
# Visualize
fig, ax = plt.subplots(figsize=(12, 6))

bars = ax.bar(range(len(adoption_size)), adoption_size['rate'].values, 
              color=COLORS['primary'], edgecolor='white')

# Add value labels
for bar, val in zip(bars, adoption_size['rate'].values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_xticks(range(len(adoption_size)))
ax.set_xticklabels(adoption_size.index, rotation=45, ha='right')
ax.set_ylabel('Adoption Rate (%)')
ax.set_xlabel('Company Size')
ax.set_title('AI Tool Adoption by Company Size', fontsize=14, fontweight='bold')
ax.set_ylim(0, 80)
ax.axhline(y=df['Is_AI_User'].mean()*100, color=COLORS['danger'], 
           linestyle='--', linewidth=2, label=f'Average ({df["Is_AI_User"].mean()*100:.1f}%)')
ax.legend(loc='upper right')

plt.tight_layout()
plt.savefig('../visualizations/02_adoption_by_company_size.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Adoption by Role

In [ ]:
# Calculate adoption by role
adoption_role = df.groupby('Role_Category')['Is_AI_User'].mean() * 100
adoption_role = adoption_role.drop('Unknown', errors='ignore')
adoption_role = adoption_role.sort_values(ascending=True)

# Visualize
fig, ax = plt.subplots(figsize=(10, 6))

colors = [COLORS['primary'] if v >= 60 else COLORS['secondary'] for v in adoption_role.values]
bars = ax.barh(range(len(adoption_role)), adoption_role.values, color=colors, edgecolor='white')

for bar, val in zip(bars, adoption_role.values):
    ax.text(val + 1, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', ha='left', va='center', fontsize=9)

ax.set_yticks(range(len(adoption_role)))
ax.set_yticklabels(adoption_role.index)
ax.set_xlabel('Adoption Rate (%)')
ax.set_title('AI Tool Adoption by Job Role', fontsize=14, fontweight='bold')
ax.set_xlim(0, 85)

plt.tight_layout()
plt.savefig('../visualizations/03_adoption_by_role.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Sentiment Analysis

In [ ]:
# Sentiment distribution
sentiment = df['Sentiment_Category'].value_counts()
print("Sentiment Distribution:")
print(sentiment)

# Pie chart
fig, ax = plt.subplots(figsize=(8, 6))

colors = [COLORS['success'], COLORS['warning'], COLORS['danger']]
ax.pie(sentiment.values, labels=sentiment.index, autopct='%1.1f%%',
       colors=colors, explode=(0.02, 0.02, 0.02), startangle=90)
ax.set_title('AI Tool Sentiment Distribution', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../visualizations/04_sentiment_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Benefits Analysis

In [ ]:
# Calculate benefits for AI users
ai_users = df[df['Is_AI_User'] == 1]

benefits = {
    'Increase Productivity': ai_users['Ben_Increase_productivit'].mean() * 100,
    'Speed Up Learning': ai_users['Ben_Speed_up_learning'].mean() * 100,
    'Greater Efficiency': ai_users['Ben_Greater_efficiency'].mean() * 100,
    'Improve Accuracy': ai_users['Ben_Improve_accuracy_in_'].mean() * 100,
    'Manageable Workload': ai_users['Ben_Make_workload_more_m'].mean() * 100,
    'Improve Collaboration': ai_users['Ben_Improve_collaboratio'].mean() * 100
}

benefits_df = pd.DataFrame.from_dict(benefits, orient='index', columns=['Percentage'])
benefits_df = benefits_df.sort_values('Percentage', ascending=True)

print("Benefits (% of AI Users):")
print(benefits_df)

In [ ]:
# Visualize benefits
fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.barh(range(len(benefits_df)), benefits_df['Percentage'].values, 
               color=COLORS['success'], edgecolor='white')

for bar, val in zip(bars, benefits_df['Percentage'].values):
    ax.text(val + 1, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', ha='left', va='center', fontsize=10, fontweight='bold')

ax.set_yticks(range(len(benefits_df)))
ax.set_yticklabels(benefits_df.index)
ax.set_xlabel('% of AI Users')
ax.set_title('Top Benefits of AI Coding Tools', fontsize=14, fontweight='bold')
ax.set_xlim(0, 100)

plt.tight_layout()
plt.savefig('../visualizations/06_benefits.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Challenges Analysis

In [ ]:
# Calculate challenges
challenges = {
    'Trust Issues': ai_users['Challenge_Trust'].mean() * 100,
    'Lacks Codebase Context': ai_users['Challenge_Context'].mean() * 100,
    'Security Concerns': ai_users['Challenge_Security'].mean() * 100,
    'Training Gaps': ai_users['Challenge_Training'].mean() * 100,
    'Uneven Adoption': ai_users['Challenge_Adoption'].mean() * 100
}

challenges_df = pd.DataFrame.from_dict(challenges, orient='index', columns=['Percentage'])
challenges_df = challenges_df.sort_values('Percentage', ascending=True)

print("Challenges (% of AI Users):")
print(challenges_df)

In [ ]:
# Visualize challenges
fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.barh(range(len(challenges_df)), challenges_df['Percentage'].values, 
               color=COLORS['danger'], edgecolor='white')

for bar, val in zip(bars, challenges_df['Percentage'].values):
    ax.text(val + 1, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', ha='left', va='center', fontsize=10, fontweight='bold')

ax.set_yticks(range(len(challenges_df)))
ax.set_yticklabels(challenges_df.index)
ax.set_xlabel('% of AI Users')
ax.set_title('Top Challenges with AI Coding Tools', fontsize=14, fontweight='bold')
ax.set_xlim(0, 70)

plt.tight_layout()
plt.savefig('../visualizations/07_challenges.png', dpi=300, bbox_inches='tight')
plt.show()

## 8. ROI Analysis

In [ ]:
# ROI Summary
print("ROI Summary:")
print(f"  Conservative (2 hrs/wk): Median = {df['ROI_Low'].median():,.0f}%")
print(f"  Moderate (5 hrs/wk): Median = {df['ROI_Mid'].median():,.0f}%")
print(f"  Optimistic (8 hrs/wk): Median = {df['ROI_High'].median():,.0f}%")

In [ ]:
# ROI by scenario chart
fig, ax = plt.subplots(figsize=(8, 5))

scenarios = ['Conservative\n(2 hrs/wk)', 'Moderate\n(5 hrs/wk)', 'Optimistic\n(8 hrs/wk)']
medians = [df['ROI_Low'].median(), df['ROI_Mid'].median(), df['ROI_High'].median()]

colors = [COLORS['warning'], COLORS['success'], COLORS['primary']]
bars = ax.bar(scenarios, medians, color=colors, edgecolor='white', linewidth=2)

for bar, val in zip(bars, medians):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            f'{val:,.0f}%', ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_ylabel('Median ROI (%)')
ax.set_title('ROI by Productivity Scenario', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../visualizations/08_roi_scenarios.png', dpi=300, bbox_inches='tight')
plt.show()

## 9. Threat Perception

In [ ]:
# Threat distribution
threat = df['Threat_Category'].value_counts()
print("Job Threat Perception:")
print(threat)

# Pie chart
fig, ax = plt.subplots(figsize=(8, 6))

colors = [COLORS['success'], COLORS['warning'], COLORS['danger']]
ax.pie(threat.values, labels=threat.index, autopct='%1.1f%%',
       colors=colors, explode=(0.02, 0.02, 0.02), startangle=90)
ax.set_title('Do Developers See AI as a Job Threat?', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../visualizations/10_threat_perception.png', dpi=300, bbox_inches='tight')
plt.show()

## 10. Comprehensive Dashboard

In [ ]:
# Create comprehensive dashboard
fig = plt.figure(figsize=(20, 16))

fig.suptitle('GenAI ROI Analysis Dashboard\nStack Overflow Developer Survey 2024', 
             fontsize=20, fontweight='bold', y=0.98)

gs = fig.add_gridspec(3, 3, hspace=0.35, wspace=0.3)

# 1. Adoption Overview
ax1 = fig.add_subplot(gs[0, 0])
adoption = df['AI_Status'].value_counts()
ax1.pie(adoption.values, labels=adoption.index, autopct='%1.1f%%', 
        colors=[COLORS['success'], COLORS['danger'], COLORS['warning']])
ax1.set_title('AI Adoption Status', fontweight='bold')

# 2. Adoption by Company Size
ax2 = fig.add_subplot(gs[0, 1])
size_adoption = df.groupby('Company_Size_Category')['Is_AI_User'].mean() * 100
size_order = ['Micro (2-9)', 'Small (10-19)', 'Small-Mid (20-99)', 'Mid (100-499)', 
              'Mid-Large (500-999)', 'Large (1K-5K)', 'Enterprise (10K+)']
size_adoption = size_adoption.reindex([s for s in size_order if s in size_adoption.index])
ax2.bar(range(len(size_adoption)), size_adoption.values, color=COLORS['primary'])
ax2.set_xticks(range(len(size_adoption)))
ax2.set_xticklabels(['Micro', 'Small', 'Sm-Mid', 'Mid', 'Mid-Lg', 'Large', 'Entrp'], 
                    rotation=45, ha='right', fontsize=8)
ax2.set_ylabel('Adoption %')
ax2.set_title('Adoption by Company Size', fontweight='bold')

# 3. Sentiment Distribution
ax3 = fig.add_subplot(gs[0, 2])
sentiment = df['Sentiment_Category'].value_counts()
ax3.pie(sentiment.values, labels=sentiment.index, autopct='%1.1f%%',
        colors=[COLORS['success'], COLORS['warning'], COLORS['danger']])
ax3.set_title('Sentiment Distribution', fontweight='bold')

# 4. Benefits
ax4 = fig.add_subplot(gs[1, 0])
benefits_vals = [ai_users['Ben_Increase_productivit'].mean() * 100,
                ai_users['Ben_Speed_up_learning'].mean() * 100,
                ai_users['Ben_Greater_efficiency'].mean() * 100,
                ai_users['Ben_Improve_accuracy_in_'].mean() * 100]
ax4.barh(['Productivity', 'Learning', 'Efficiency', 'Accuracy'], benefits_vals, color=COLORS['success'])
ax4.set_xlabel('% of AI Users')
ax4.set_title('Top Benefits', fontweight='bold')
ax4.set_xlim(0, 100)

# 5. Challenges
ax5 = fig.add_subplot(gs[1, 1])
challenges_vals = [ai_users['Challenge_Trust'].mean() * 100,
                  ai_users['Challenge_Context'].mean() * 100,
                  ai_users['Challenge_Security'].mean() * 100,
                  ai_users['Challenge_Training'].mean() * 100]
ax5.barh(['Trust Issues', 'Lacks Context', 'Security', 'Training'], challenges_vals, color=COLORS['danger'])
ax5.set_xlabel('% of AI Users')
ax5.set_title('Top Challenges', fontweight='bold')
ax5.set_xlim(0, 70)

# 6. ROI Scenarios
ax6 = fig.add_subplot(gs[1, 2])
scenarios = ['Low\n(2hr/wk)', 'Mid\n(5hr/wk)', 'High\n(8hr/wk)']
roi_medians = [df['ROI_Low'].median(), df['ROI_Mid'].median(), df['ROI_High'].median()]
ax6.bar(scenarios, roi_medians, color=[COLORS['warning'], COLORS['success'], COLORS['primary']])
ax6.set_ylabel('Median ROI %')
ax6.set_title('ROI by Scenario', fontweight='bold')

# 7. Adoption by Role
ax7 = fig.add_subplot(gs[2, 0:2])
role_adoption = df.groupby('Role_Category')['Is_AI_User'].mean() * 100
role_adoption = role_adoption.drop('Unknown', errors='ignore').sort_values(ascending=True)
ax7.barh(range(len(role_adoption)), role_adoption.values, color=COLORS['primary'])
ax7.set_yticks(range(len(role_adoption)))
ax7.set_yticklabels(role_adoption.index, fontsize=9)
ax7.set_xlabel('Adoption Rate %')
ax7.set_title('Adoption by Job Role', fontweight='bold')

# 8. Threat Perception
ax8 = fig.add_subplot(gs[2, 2])
threat = df['Threat_Category'].value_counts()
ax8.pie(threat.values, labels=threat.index, autopct='%1.1f%%',
        colors=[COLORS['success'], COLORS['warning'], COLORS['danger']])
ax8.set_title('Job Threat Perception', fontweight='bold')

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('../visualizations/12_dashboard.png', dpi=300, bbox_inches='tight')
plt.show()

## 11. Key Insights Summary

In [ ]:
print("="*60)
print("KEY INSIGHTS SUMMARY")
print("="*60)

print(f"\n📊 ADOPTION")
print(f"   - Overall adoption rate: {df['Is_AI_User'].mean()*100:.1f}%")
print(f"   - Highest adoption: Small companies (2-9 employees)")
print(f"   - Lowest adoption: Enterprise (10K+ employees)")

print(f"\n💭 SENTIMENT")
print(f"   - Positive sentiment: {(df['Sentiment_Category']=='Positive').sum()/len(df)*100:.1f}%")
print(f"   - Negative sentiment: {(df['Sentiment_Category']=='Negative').sum()/len(df)*100:.1f}%")

print(f"\n✅ TOP BENEFITS")
print(f"   1. Productivity: {ai_users['Ben_Increase_productivit'].mean()*100:.1f}%")
print(f"   2. Learning: {ai_users['Ben_Speed_up_learning'].mean()*100:.1f}%")
print(f"   3. Efficiency: {ai_users['Ben_Greater_efficiency'].mean()*100:.1f}%")

print(f"\n⚠️ TOP CHALLENGES")
print(f"   1. Trust Issues: {ai_users['Challenge_Trust'].mean()*100:.1f}%")
print(f"   2. Lacks Context: {ai_users['Challenge_Context'].mean()*100:.1f}%")
print(f"   3. Security: {ai_users['Challenge_Security'].mean()*100:.1f}%")

print(f"\n💰 ROI POTENTIAL")
print(f"   - Median ROI (5 hrs/wk saved): {df['ROI_Mid'].median():,.0f}%")

print(f"\n😰 THREAT PERCEPTION")
print(f"   - Not threatened: {(df['Threat_Category']=='Not Threatened').sum()/df['Threat_Category'].notna().sum()*100:.1f}%")
print(f"   - Threatened: {(df['Threat_Category']=='Threatened').sum()/df['Threat_Category'].notna().sum()*100:.1f}%")

In [ ]:
print("\n" + "="*60)
print("✅ ANALYSIS COMPLETE!")
print("="*60)
print("\nAll visualizations saved to ../visualizations/")
print("\nReady for portfolio presentation!")